# 00 · Extract outcome cohorts from code definitions

Flags every patient in the cohort for each outcome defined in `definitions/`.

### Adding a definition
Put `.txt` files in `definitions/<set_name>/`, named `<outcome>_<suffix>.txt`. The outcome name is everything before the
**last** underscore, so `ami_icd9.txt` and `ami_icd10.txt` are combined into one `ami` outcome.

One code per line:

| Line | Meaning |
|---|---|
| `I21.4` | exact match |
| `I63.` or `I63*` | prefix match: every code starting with `I63` |
| blank / `# comment` | ignored |

Codes are matched as strings, regardless of ICD version, because the source table has no version column.

### Outputs, in `outputs/<set_name>/`
- `events.parquet`: every ICD row that matched an outcome (long format)
- `patients.parquet`: one row per cohort patient, with flags and timing for each outcome
- `code_audit.csv`: which data codes each definition line matched, with row and patient counts

In [ ]:
from pathlib import Path

import polars as pl

pl.Config.set_tbl_rows(50)
pl.Config.set_fmt_str_lengths(80)

## Parameters

In [ ]:
# True runs on the synthetic tables in examples/ (see examples/README.md for the expected input format)
USE_EXAMPLE_DATA = True

if USE_EXAMPLE_DATA:
    ICD_PATH = Path("examples/icd.csv")
    COHORT_PATH = Path("examples/cohort.csv")
    OUTPUT_DIR = Path("outputs/examples")
    COLUMN_MAP = {}
else:
    ICD_PATH = Path("../CLINICAL/ANON_ICD_20261118.parquet")
    COHORT_PATH = Path(
        "../CLINICAL/ANON_FOLLUP_2026-11-23.parquet"
    )  # one row per patient: the denominator
    OUTPUT_DIR = Path("outputs")
    # Rename your columns to the pipeline's names: {your_name: pipeline_name}. Columns not listed are left as is.
    COLUMN_MAP = {"empi_anon": "patient_id", "acc_anon": "accession_id"}

DEFINITIONS_DIR = Path("definitions")

# None runs every subfolder of DEFINITIONS_DIR; or list specific sets, e.g. ["dapamede2026_ehj_bac"]
DEFINITION_SETS = None

# days_to_icd is relative to the index exam. Events before this day are "prevalent", events on or after it are "incident".
INDEX_DAY = 0

KEYS = ["patient_id", "accession_id"]

## Definition loading

In [ ]:
def parse_definition_file(path: Path) -> list[dict]:
    # One dict per code line. A trailing '*' or '.' marks a prefix match.
    rows = []
    for line_no, raw in enumerate(path.read_text().splitlines(), start=1):
        entry = raw.split("#", 1)[0].strip().upper()
        if not entry:
            continue
        if entry.endswith("*"):
            pattern, match_type = entry.rstrip("*"), "prefix"
        elif entry.endswith("."):
            pattern, match_type = entry, "prefix"
        else:
            pattern, match_type = entry, "exact"
        rows.append(
            {
                "source_file": path.name,
                "line": line_no,
                "pattern": pattern,
                "match_type": match_type,
            }
        )
    return rows


def outcome_name(path: Path) -> str:
    # 'ami_icd10' -> 'ami', 'heart_failure_icd9' -> 'heart_failure', 'stroke' -> 'stroke'
    return path.stem.rsplit("_", 1)[0] if "_" in path.stem else path.stem


def load_definition_set(set_dir: Path) -> pl.DataFrame:
    rows = []
    for path in sorted(set_dir.glob("*.txt")):
        parsed = parse_definition_file(path)
        if not parsed:
            print(f"WARNING: {path} has no codes, skipping")
        rows += [{"outcome": outcome_name(path), **r} for r in parsed]
    if not rows:
        raise ValueError(f"No definition codes found in {set_dir}")
    return pl.DataFrame(rows)

In [ ]:
set_names = DEFINITION_SETS or sorted(
    p.name
    for p in DEFINITIONS_DIR.iterdir()
    if p.is_dir() and not p.name.startswith(".")
)
definitions = {s: load_definition_set(DEFINITIONS_DIR / s) for s in set_names}

for s, defs in definitions.items():
    print(f"== {s}")
    display(
        defs.group_by("outcome")
        .agg(
            files=pl.col("source_file").unique().sort().str.join(", "),
            n_exact=(pl.col("match_type") == "exact").sum(),
            n_prefix=(pl.col("match_type") == "prefix").sum(),
        )
        .sort("outcome")
    )

## Load data

In [ ]:
def read_table(path: Path, **csv_kwargs) -> pl.DataFrame:
    # Parquet or CSV, with columns renamed to the pipeline's names
    df = (
        pl.read_csv(path, **csv_kwargs)
        if path.suffix == ".csv"
        else pl.read_parquet(path)
    )
    return df.rename(COLUMN_MAP, strict=False)


# icd_code must stay a string: as a number, '410.90' would become 410.9
icd = read_table(ICD_PATH, schema_overrides={"icd_code": pl.String}).select(
    *KEYS, "icd_code", "days_to_icd"
)
cohort = read_table(COHORT_PATH)

# Normalize the distinct codes only (about 39k), not all 37M rows; we join back on the raw code.
data_codes = (
    icd.drop_nulls("icd_code")
    .group_by("icd_code")
    .agg(n_rows=pl.len(), n_patients=pl.col("patient_id").n_unique())
    .with_columns(code_norm=pl.col("icd_code").str.strip_chars().str.to_uppercase())
)

icd_pts = icd.select(KEYS).unique()
print(
    f"ICD rows: {icd.height:,} | distinct codes: {data_codes.height:,} | patients: {icd_pts.height:,}"
)
print(f"Cohort patients: {cohort.height:,}")
print(
    f"  cohort patients with no ICD rows (kept as outcome-negative): {cohort.join(icd_pts, on=KEYS, how='anti').height:,}"
)
print(
    f"  ICD patients not in cohort (dropped): {icd_pts.join(cohort, on=KEYS, how='anti').height:,}"
)
print(
    f"  ICD rows with null days_to_icd (count toward 'any', not toward timing): {icd['days_to_icd'].null_count():,}"
)

## Pipeline

In [ ]:
def match_codes(defs: pl.DataFrame, data_codes: pl.DataFrame) -> pl.DataFrame:
    # One row per (definition line, matching data code).
    exact = defs.filter(pl.col("match_type") == "exact").join(
        data_codes, left_on="pattern", right_on="code_norm", how="inner"
    )
    prefix = (
        defs.filter(pl.col("match_type") == "prefix")
        .join(data_codes, how="cross")
        .filter(pl.col("code_norm").str.starts_with(pl.col("pattern")))
        .drop("code_norm")
    )
    return pl.concat([exact, prefix], how="diagonal").sort(
        "outcome", "source_file", "line", "icd_code"
    )


def build_audit(defs: pl.DataFrame, matches: pl.DataFrame) -> pl.DataFrame:
    # Every definition line, including lines that matched nothing in the data.
    return defs.join(
        matches.select(
            "outcome", "source_file", "line", "icd_code", "n_rows", "n_patients"
        ),
        on=["outcome", "source_file", "line"],
        how="left",
    ).sort("outcome", "source_file", "line", "icd_code")


def summarize_patients(
    events: pl.DataFrame, cohort: pl.DataFrame, outcomes: list[str]
) -> pl.DataFrame:
    # Wide table: one row per cohort patient, with columns <outcome>_<metric>.
    day = pl.col("days_to_icd")
    per = events.group_by(*KEYS, "outcome").agg(
        any=pl.lit(True),
        prevalent=(day < INDEX_DAY).any(),
        incident=(day >= INDEX_DAY).any(),
        days_to_first_incident=day.filter(day >= INDEX_DAY).min(),
        days_to_last_prevalent=day.filter(day < INDEX_DAY).max(),
        n_events=pl.len(),
    )
    out = cohort
    for o in outcomes:
        wide = per.filter(pl.col("outcome") == o).drop("outcome")
        wide = wide.rename({c: f"{o}_{c}" for c in wide.columns if c not in KEYS})
        out = out.join(wide, on=KEYS, how="left").with_columns(
            pl.col(f"{o}_any", f"{o}_prevalent", f"{o}_incident").fill_null(False),
            pl.col(f"{o}_n_events").fill_null(0),
        )
    return out


def outcome_counts(patients: pl.DataFrame, outcomes: list[str]) -> pl.DataFrame:
    n = patients.height
    rows = []
    for o in outcomes:
        prev, inc = pl.col(f"{o}_prevalent"), pl.col(f"{o}_incident")
        r = patients.select(
            any=pl.col(f"{o}_any").sum(),
            prevalent=prev.sum(),
            incident=inc.sum(),
            incident_without_prevalent=(inc & ~prev).sum(),
        ).row(0, named=True)
        rows.append({"outcome": o, **r, "pct_any": round(100 * r["any"] / n, 2)})
    return pl.DataFrame(rows)


def run_definition_set(set_name: str, defs: pl.DataFrame) -> dict[str, pl.DataFrame]:
    outcomes = sorted(defs["outcome"].unique())
    matches = match_codes(defs, data_codes)
    code_map = matches.select(
        "outcome", "icd_code"
    ).unique()  # a code can match several lines

    events = (
        icd.join(code_map, on="icd_code", how="inner")
        .join(cohort.select(KEYS), on=KEYS, how="semi")
        .select(*KEYS, "outcome", "icd_code", "days_to_icd")
        .sort(*KEYS, "outcome", "days_to_icd")
    )
    patients = summarize_patients(events, cohort, outcomes)
    audit = build_audit(defs, matches)

    out_dir = OUTPUT_DIR / set_name
    out_dir.mkdir(parents=True, exist_ok=True)
    events.write_parquet(out_dir / "events.parquet")
    patients.write_parquet(out_dir / "patients.parquet")
    audit.write_csv(out_dir / "code_audit.csv")

    return {
        "events": events,
        "patients": patients,
        "audit": audit,
        "counts": outcome_counts(patients, outcomes),
    }

## Run

In [ ]:
results = {}
for s, defs in definitions.items():
    results[s] = run_definition_set(s, defs)
    print(f"== {s}  ->  {OUTPUT_DIR / s}")
    display(results[s]["counts"])

## QA

Check each set before using its outputs:
1. **Unmatched lines**: definition codes that never occur in the data. These are often typos, or a code was written
   without its dot.
2. **Near misses**: data codes in the same 3-character category as a definition code that were *not* included
   (e.g. `I21.A1` next to `I21.x`). Confirm that each exclusion is intentional.

In [ ]:
def near_misses(defs: pl.DataFrame, audit: pl.DataFrame) -> pl.DataFrame:
    # Data codes that share a category (the part before '.') with an outcome's codes but aren't in the definition.
    cats = defs.select(
        "outcome", category=pl.col("pattern").str.split(".").list.first()
    ).unique()
    matched = audit.drop_nulls("icd_code").select("outcome", "icd_code").unique()
    return (
        data_codes.with_columns(
            category=pl.col("code_norm").str.split(".").list.first()
        )
        .join(cats, on="category", how="inner")
        .join(matched, on=["outcome", "icd_code"], how="anti")
        .select("outcome", "icd_code", "n_rows", "n_patients")
        .sort("outcome", "icd_code")
    )


for s, res in results.items():
    print(f"== {s}")
    print("Unmatched definition lines:")
    display(
        res["audit"]
        .filter(pl.col("icd_code").is_null())
        .select("outcome", "source_file", "line", "pattern")
    )
    print("Near misses (same category, not in definition):")
    display(near_misses(definitions[s], res["audit"]))

In [ ]:
# Codes matched per outcome, ranked by number of patients
for s, res in results.items():
    print(f"== {s}")
    display(
        res["audit"]
        .drop_nulls("icd_code")
        .unique(["outcome", "icd_code"])
        .select("outcome", "icd_code", "match_type", "pattern", "n_rows", "n_patients")
        .sort("outcome", "n_patients", descending=[False, True])
    )